# 02: Desarrollo y Pruebas del Servidor MCP

Este notebook se utiliza para validar las herramientas expuestas por `cognito_mcp_server.py`.

**Pruebas realizadas:**
1. **Test de Herramientas Individuales:**
   - `generate_with_llm`: Verifica la conexión con Ollama y la generación de texto.
   - `query_vector_db`: Valida la creación de embeddings y la búsqueda en Qdrant.
2. **Test del Workflow RAG Completo:**
   - `execute_rag_pipeline`: Prueba el flujo completo de recuperación y generación para asegurar que la orquestación funciona como se espera.

In [ ]:
# Celda 1: Configuración e importaciones

import asyncio
import json
import sys
import os

# Añadir el directorio del servidor MCP al path para poder importar sus módulos
mcp_server_path = os.path.abspath('../mcp-server')
if mcp_server_path not in sys.path:
    sys.path.append(mcp_server_path)

# Importar las herramientas directamente desde el script del servidor
from cognito_mcp_server import (
    generate_with_llm,
    query_vector_db,
    execute_rag_pipeline,
    OLLAMA_URL,  # Importar para verificar
    QDRANT_URL   # Importar para setup
)

from qdrant_client import QdrantClient, models
import httpx

print("Módulos y herramientas importados correctamente.")

In [ ]:
# Celda 2: Setup de Datos para Pruebas en Qdrant

TEST_COLLECTION_NAME = "mcp_dev_test_collection"

async def setup_qdrant_for_test():
    """Prepara una colección en Qdrant con un documento de ejemplo."""
    print("Preparando Qdrant para las pruebas...")
    try:
        qdrant_client = QdrantClient(url=QDRANT_URL)
        async with httpx.AsyncClient(timeout=30.0) as client:
            # Generar embedding para el documento de prueba
            test_doc = "Tarragona está explorando el uso de IA en servicios urbanos."
            embedding_response = await client.post(
                f"{OLLAMA_URL}/api/embeddings",
                json={"model": "nomic-embed-text", "prompt": test_doc}
            )
            embedding_response.raise_for_status()
            embedding = embedding_response.json()["embedding"]
            vector_size = len(embedding)

            # Recrear la colección
            qdrant_client.recreate_collection(
                collection_name=TEST_COLLECTION_NAME,
                vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE)
            )

            # Insertar el documento de prueba
            qdrant_client.upsert(
                collection_name=TEST_COLLECTION_NAME,
                points=[models.PointStruct(id=1, vector=embedding, payload={"text": test_doc})],
                wait=True
            )
        print(f"✓ Qdrant listo. Colección '{TEST_COLLECTION_NAME}' preparada.")
        return True
    except Exception as e:
        print(f"❌ Error al preparar Qdrant: {e}")
        return False

# Ejecutar la preparación
qdrant_ready = await setup_qdrant_for_test()

In [ ]:
# Celda 3: Test de Herramientas Individuales

async def test_ollama_tool():
    """Valida la herramienta 'generate_with_llm'."""
    print("--- Probando generate_with_llm ---")
    try:
        response = await generate_with_llm(
            prompt="Explica qué es un LLM en una frase corta.",
            model="llama3.2"
        )
        assert len(response) > 10
        print(f"✓ Respuesta de Ollama: '{response[:80]}...'\n")
    except Exception as e:
        print(f"❌ Falló la prueba de Ollama: {e}\n")

async def test_qdrant_tool():
    """Valida la herramienta 'query_vector_db'."""
    print("--- Probando query_vector_db ---")
    if not qdrant_ready:
        print("- Saltando prueba de Qdrant porque la preparación falló.\n")
        return
    
    try:
        result_json = await query_vector_db(
            query="servicios para la ciudad",
            collection=TEST_COLLECTION_NAME
        )
        results = json.loads(result_json)
        assert len(results) > 0
        assert "IA" in results[0]['payload']['text']
        print(f"✓ Qdrant encontró {len(results)} documento(s) relevante(s).")
        print(f"  Mejor resultado: {results[0]['payload']['text']}\n")
    except Exception as e:
        print(f"❌ Falló la prueba de Qdrant: {e}\n")

# Ejecutar pruebas individuales
await test_ollama_tool()
await test_qdrant_tool()

In [ ]:
# Celda 4: Test del Workflow RAG Completo

async def test_complete_rag_workflow():
    """Valida la orquestación del pipeline RAG de principio a fin."""
    print("--- Probando execute_rag_pipeline ---")
    if not qdrant_ready:
        print("- Saltando prueba de RAG porque la preparación de Qdrant falló.")
        return
        
    query = "¿Cómo usa Tarragona la inteligencia artificial?"
    print(f"Pregunta: '{query}'")
    
    try:
        # En el notebook, necesitamos renombrar la colección usada por la herramienta RAG
        # para que coincida con nuestra colección de prueba.
        original_query_vector_db = query_vector_db
        
        async def mocked_query_vector_db(query, collection):
            return await original_query_vector_db(query, collection=TEST_COLLECTION_NAME)
        
        # Monkey-patch temporal de la función para la prueba
        import cognito_mcp_server
        cognito_mcp_server.query_vector_db = mocked_query_vector_db
        
        response = await execute_rag_pipeline(
            query=query
        )
        
        # Restaurar la función original
        cognito_mcp_server.query_vector_db = original_query_vector_db

        assert len(response) > 20
        assert "servicios urbanos" in response.lower() or "ia" in response.lower()
        print(f"\n✓ Respuesta del pipeline RAG:\n---\n{response}\n---")
        
    except Exception as e:
        print(f"❌ Falló la prueba del pipeline RAG: {e}")

# Ejecutar prueba del workflow completo
await test_complete_rag_workflow()

## Conclusión de las Pruebas

Si todas las celdas anteriores se han ejecutado sin errores, se puede concluir que:

1. El servidor MCP puede conectarse correctamente a **Ollama** y **Qdrant**.
2. Las herramientas `generate_with_llm` y `query_vector_db` son funcionales.
3. La lógica de orquestación en `execute_rag_pipeline` funciona como se esperaba, combinando la recuperación de información con la generación de texto.